In [ ]:
# CELL 20 — Full build setup: paths, chunk plan, and the v2 generator (identical to CELL 9; only paths, split rule and job list changed)
!pip install -q pikepdf pymupdf pypdf pebble

import os, io, re, glob, json, zlib, shutil, random, logging
import pandas as pd
import pikepdf, pypdf
import pymupdf as fitz
from pypdf import PdfWriter
from pypdf.generic import ArrayObject, StreamObject, NameObject
from PIL import Image, ImageDraw, ImageFont
from pebble import ProcessPool
from concurrent.futures import TimeoutError, as_completed
from tqdm.auto import tqdm

logging.getLogger("pypdf").setLevel(logging.ERROR)
logging.getLogger("pikepdf").setLevel(logging.ERROR)

# ---------- paths ----------
hits = sorted(glob.glob("/kaggle/input/**/clean_pool.csv", recursive=True))
assert hits, "clean_pool.csv not found - is PDP BPTP attached as input?"
CLEAN_POOL = hits[0]
SRC = os.path.join(os.path.dirname(CLEAN_POOL), "govdocs_pdfs")
assert os.path.isdir(SRC), f"govdocs_pdfs not found next to {CLEAN_POOL}"
WORK = "/kaggle/working/full"
OUT = f"{WORK}/tmp_pdfs"      # PDFs of the current chunk only (deleted after extraction)
META = f"{WORK}/meta"         # one small JSON per original (manifest rows)
CHUNKS = f"{WORK}/chunks"     # per-chunk results: features CSV + graphs file
for d in (OUT, META, CHUNKS):
    os.makedirs(d, exist_ok=True)

TIME_LIMIT = 180
CHUNK_SIZE = 300
WORKERS = 4
FAMILIES = ["acrobat distiller windows", "acrobat distiller for", "acrobat pdfwriter for",
            "adobe pdf library", "corel pdf engine"]
TOOLS = ["pikepdf", "pymupdf", "pypdf"]

LIT = re.compile(rb"\((?:[^()\\]|\\.)*\)", re.S)
TAIL = re.compile(rb"(?:\s+|-?(?:\d+\.?\d*|\.\d+)(?=[\s(\]])|\((?:[^()\\]|\\.)*\))*\s*(?:\]\s*TJ|Tj|'|\")")
VALUE = re.compile(rb"\d[\d,\.]*\d")

def make_stamp(text):
    img = Image.new("RGBA", (420, 140), (255, 255, 255, 0))
    dr = ImageDraw.Draw(img)
    dr.rectangle([4, 4, 415, 135], outline=(200, 0, 0, 255), width=8)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", 60)
    except Exception:
        font = ImageFont.load_default()
    dr.text((24, 34), text, fill=(200, 0, 0, 255), font=font)
    buf = io.BytesIO()
    img.save(buf, "PNG")
    return buf.getvalue()

PNG_HONEST = make_stamp("RECEIVED")
PNG_FAKE = make_stamp("PAID")

def digit_positions(seg):
    pos, i, n = [], 1, len(seg) - 1
    while i < n:
        b = seg[i]
        if b == 0x5C:
            j = i + 1
            if j < n and 48 <= seg[j] <= 55:
                k = j
                while k < n and k < j + 3 and 48 <= seg[k] <= 55:
                    k += 1
                i = k
            else:
                i = j + 1
            continue
        if 48 <= b <= 57:
            pos.append(i)
        i += 1
    return pos

def candidates(data):
    digits, values = [], []
    for m in LIT.finditer(data):
        if not TAIL.match(data[m.end():m.end() + 300]):
            continue
        seg = m.group()
        digits += [m.start() + p for p in digit_positions(seg)]
        for v in VALUE.finditer(seg):
            if v.start() > 0 and seg[v.start() - 1] == 0x5C:
                continue
            if sum(c in b"0123456789" for c in v.group()) >= 3:
                values.append((m.start() + v.start(), m.start() + v.end()))
    return digits, values

def new_value(old, rng):
    s = old.decode("latin-1")
    two_dec = s.count(".") == 1 and len(s.split(".")[1]) == 2
    intpart = s.split(".")[0] if two_dec else s
    nd = max(1, sum(c.isdigit() for c in intpart) + rng.choice([-1, 1, 2]))
    n = rng.randint(10 ** (nd - 1), 10 ** nd - 1)
    out = (f"{n:,}" if "," in s else str(n)) + (f".{rng.randint(0, 99):02d}" if two_dec else "")
    if out == s:
        out = str(n + 1)
    return out.encode()

def pk_streams(pdf, i):
    c = pdf.pages[i].obj.get("/Contents")
    if c is None:
        return []
    return [s for s in (list(c) if isinstance(c, pikepdf.Array) else [c]) if isinstance(s, pikepdf.Stream)]

def choose_specs(src, rng):
    dspec = vspec = None
    with pikepdf.open(src) as p:
        order = list(range(len(p.pages)))
        rng.shuffle(order)
        for i in order:
            for k, s in enumerate(pk_streams(p, i)):
                try:
                    data = s.read_bytes()
                except Exception:
                    continue
                d, v = candidates(data)
                if dspec is None and d:
                    pos = rng.choice(d)
                    old = data[pos:pos + 1]
                    new = bytes([rng.choice([c for c in b"0123456789" if c != old[0]])])
                    dspec = dict(page=i, k=k, start=pos, end=pos + 1, old=old, new=new)
                if vspec is None and v:
                    st, en = rng.choice(v)
                    vspec = dict(page=i, k=k, start=st, end=en, old=data[st:en], new=new_value(data[st:en], rng))
                if dspec and vspec:
                    return dspec, vspec
    return dspec, vspec

def apply_spec(data, spec):
    if data[spec["start"]:spec["end"]] != spec["old"]:
        raise RuntimeError("edit location mismatch")
    return data[:spec["start"]] + spec["new"] + data[spec["end"]:]

def edit_pikepdf(pdf, spec):
    s = pk_streams(pdf, spec["page"])[spec["k"]]
    s.write(apply_spec(s.read_bytes(), spec))

def edit_fitz(doc, spec):
    x = doc[spec["page"]].get_contents()[spec["k"]]
    doc.update_stream(x, apply_spec(doc.xref_stream(x), spec))

def py_streams(page):
    c = page.get("/Contents")
    if c is None:
        return []
    c = c.get_object()
    out = []
    for it in (list(c) if isinstance(c, ArrayObject) else [c]):
        o = it.get_object()
        if isinstance(o, StreamObject):
            out.append(o)
    return out

def edit_pypdf(w, spec):
    o = py_streams(w.pages[spec["page"]])[spec["k"]]
    new = apply_spec(o.get_data(), spec)
    flt = o.get("/Filter")
    if flt is not None and flt != "/FlateDecode" and list(flt) != ["/FlateDecode"]:
        o[NameObject("/Filter")] = NameObject("/FlateDecode")
    if "/DecodeParms" in o:
        del o["/DecodeParms"]
    o.set_data(new)

def run_tool(tool, src, out, spec, mode):
    if tool == "pikepdf":
        with pikepdf.open(src) as p:
            if spec:
                edit_pikepdf(p, spec)
            if mode == "recompress":
                p.save(out, compress_streams=True, recompress_flate=True,
                       object_stream_mode=pikepdf.ObjectStreamMode.generate)
            elif mode == "linearize":
                p.save(out, linearize=True)
            else:
                p.save(out)
    elif tool == "pymupdf":
        d = fitz.open(src)
        if spec:
            edit_fitz(d, spec)
        if mode == "recompress":
            for pg in d:
                for x in pg.get_contents():
                    d.update_stream(x, d.xref_stream(x))
            d.save(out, garbage=4, deflate=True)
        else:
            d.save(out)
        d.close()
    else:
        w = PdfWriter(clone_from=src)
        if spec:
            edit_pypdf(w, spec)
        if mode == "recompress":
            for pg in w.pages:
                pg.compress_content_streams()
        with open(out, "wb") as f:
            w.write(f)

def fitz_incr(src, out, action):
    shutil.copy(src, out)
    d = fitz.open(out)
    try:
        if d.is_repaired:
            raise RuntimeError("file needed repair")
        res = action(d)
        d.saveIncr()
        return res
    finally:
        d.close()

def fitz_full(src, out, action):
    d = fitz.open(src)
    try:
        res = action(d)
        d.save(out)
        return res
    finally:
        d.close()

def add_note(d, i):
    r = d[i].rect
    d[i].add_text_annot(fitz.Point(r.x0 + 36, r.y0 + 36), "Reviewed")

def add_stamp(d, i, png):
    r = d[i].rect
    d[i].insert_image(fitz.Rect(r.x1 - 210, r.y0 + 40, r.x1 - 40, r.y0 + 97), stream=png, overlay=True)

def add_textbox(d, i, text):
    r = d[i].rect
    d[i].add_freetext_annot(fitz.Rect(r.x0 + 40, r.y1 - 80, r.x0 + 300, r.y1 - 58), text, fontsize=11)

def cover_replace(d, rng):
    order = list(range(len(d)))
    rng.shuffle(order)
    for i in order:
        words = [w for w in d[i].get_text("words") if re.search(r"\d", w[4]) and len(w[4]) >= 2]
        if not words:
            continue
        x0, y0, x1, y1, txt = rng.choice(words)[:5]
        r = fitz.Rect(x0, y0, x1, y1)
        if r.is_empty or r.height < 3:
            continue
        new = "".join(rng.choice("0123456789") if c.isdigit() else c for c in txt)
        if new == txt:
            new = txt[:-1] + ("7" if txt[-1] != "7" else "3")
        d[i].draw_rect(r, color=None, fill=(1, 1, 1), overlay=True)
        d[i].insert_text(fitz.Point(x0, y1 - r.height * 0.2), new, fontsize=max(5, r.height * 0.8),
                         fontname="helv", color=(0, 0, 0))
        return {"page": i, "old": txt, "new": new}
    raise RuntimeError("no numeric word for cover-and-replace")

def cleanup(stem, keep_meta=False):
    for p in glob.glob(os.path.join(OUT, f"{stem}__*.pdf")):
        os.remove(p)
    mp = os.path.join(META, stem + ".json")
    if not keep_meta and os.path.exists(mp):
        os.remove(mp)

def split_of(name):
    return "dev" if zlib.crc32(name.encode()) % 100 < 15 else "train"

# ---------- one original -> 15 versions (same as CELL 9; only the split line differs) ----------
def process(job):
    name, fam = job
    fitz.TOOLS.mupdf_display_errors(False)
    stem = name[:-4]
    src = os.path.join(SRC, name)
    meta_path = os.path.join(META, stem + ".json")
    if os.path.exists(meta_path):
        with open(meta_path) as fh:
            rows = json.load(fh)
        if all(os.path.exists(os.path.join(OUT, r["out_file"])) for r in rows):
            return name, rows, None
    cleanup(stem)
    try:
        rng = random.Random(zlib.crc32(name.encode()) + 2026)
        split = split_of(name)
        t_main, t_comp = rng.choice(TOOLS), rng.choice(TOOLS)
        dspec, vspec = choose_specs(src, rng)
        if dspec is None or vspec is None:
            raise RuntimeError("no editable number")
        P = lambda v: os.path.join(OUT, f"{stem}__{v}.pdf")
        pg = dspec["page"]

        shutil.copy(src, P("G0_original"))
        run_tool(t_main, src, P("G1_resave"), None, "resave")
        run_tool(t_comp, src, P("G2_recompress"), None, "recompress")
        fitz_incr(src, P("G3_note_incr"), lambda d: add_note(d, pg))
        run_tool("pikepdf", src, P("G4_linearize"), None, "linearize")
        fitz_full(src, P("G5_stamp_honest"), lambda d: add_stamp(d, pg, PNG_HONEST))
        fitz_incr(src, P("G6_textbox_honest"), lambda d: add_textbox(d, pg, "Approved - HR dept."))

        run_tool(t_main, src, P("F1_digit_resave"), dspec, "resave")
        run_tool(t_comp, src, P("F2_digit_recompress"), dspec, "recompress")
        fitz_incr(src, P("F3_digit_incr"), lambda d: edit_fitz(d, dspec))
        run_tool(t_main, src, P("F4_value_resave"), vspec, "resave")
        cov = fitz_full(src, P("F5_cover_replace"), lambda d: cover_replace(d, rng))
        run_tool("pikepdf", src, P("F6_digit_linearize"), dspec, "linearize")
        fitz_full(src, P("F7_stamp_fake"), lambda d: add_stamp(d, pg, PNG_FAKE))
        fitz_incr(src, P("F8_textbox_fake"), lambda d: add_textbox(d, pg, "TOTAL DUE: $98,450.00"))

        dold, dnew = dspec["old"].decode("latin-1"), dspec["new"].decode("latin-1")
        vold, vnew = vspec["old"].decode("latin-1"), vspec["new"].decode("latin-1")
        spec = [
            ("G0_original", 0, "none", "original", "none", None, None, None, "G0"),
            ("G1_resave", 0, t_main, "full", "none", None, None, None, "G1"),
            ("G2_recompress", 0, t_comp, "full_recompress", "none", None, None, None, "G2"),
            ("G3_note_incr", 0, "pymupdf", "incremental", "note", pg, None, None, "G3"),
            ("G4_linearize", 0, "pikepdf", "linearize", "none", None, None, None, "G4"),
            ("G5_stamp_honest", 0, "pymupdf", "full", "stamp", pg, None, None, "G5"),
            ("G6_textbox_honest", 0, "pymupdf", "incremental", "textbox", pg, None, None, "G6"),
            ("F1_digit_resave", 1, t_main, "full", "digit", pg, dold, dnew, "G1"),
            ("F2_digit_recompress", 1, t_comp, "full_recompress", "digit", pg, dold, dnew, "G2"),
            ("F3_digit_incr", 1, "pymupdf", "incremental", "digit", pg, dold, dnew, "G3"),
            ("F4_value_resave", 1, t_main, "full", "value", vspec["page"], vold, vnew, "G1"),
            ("F5_cover_replace", 1, "pymupdf", "full", "cover", cov["page"], cov["old"], cov["new"], "G1"),
            ("F6_digit_linearize", 1, "pikepdf", "linearize", "digit", pg, dold, dnew, "G4"),
            ("F7_stamp_fake", 1, "pymupdf", "full", "stamp", pg, None, None, "G5"),
            ("F8_textbox_fake", 1, "pymupdf", "incremental", "textbox", pg, None, None, "G6"),
        ]
        rows = [{"out_file": f"{stem}__{v}.pdf", "orig_file": name, "family": fam, "variant": v,
                 "label": lab, "tool": tool, "save_mode": mode, "edit_type": et, "edit_page": ep,
                 "old_value": ov, "new_value": nv, "twin": tw, "split": split}
                for v, lab, tool, mode, et, ep, ov, nv, tw in spec]
        with open(meta_path, "w") as fh:
            json.dump(rows, fh)
        return name, rows, None
    except Exception as ex:
        cleanup(stem)
        return name, None, f"{type(ex).__name__}: {str(ex)[:50]}"

# ---------- job list and chunk plan ----------
pool = pd.read_csv(CLEAN_POOL)
pool = pool[pool.producer_family.isin(FAMILIES)].copy()
pool["fam_order"] = pool.producer_family.map({f: i for i, f in enumerate(FAMILIES)})
pool = pool.sort_values(["fam_order", "file"])
JOBS = list(zip(pool.file, pool.producer_family))
CHUNK_LIST = [JOBS[i:i + CHUNK_SIZE] for i in range(0, len(JOBS), CHUNK_SIZE)]

print("Originals dir:", SRC, "|", len(glob.glob(SRC + "/*.pdf")), "PDFs")
print("\n=== ORIGINALS PER FAMILY (all clean files, zips 000-019) ===")
print(pool.producer_family.value_counts().reindex(FAMILIES).to_string())
print("Total originals:", len(JOBS), "| expected files if all succeed:", len(JOBS) * 15)
print("Dev share (originals):", round(sum(split_of(n) == "dev" for n, _ in JOBS) / len(JOBS) * 100, 1), "%")
print("\nChunks:", len(CHUNK_LIST), "of up to", CHUNK_SIZE, "originals")
print("Chunks already finished:", len(glob.glob(f"{CHUNKS}/chunk_*_features.csv")))
print("\nLibrary versions -> pikepdf", pikepdf.__version__, "| pymupdf", fitz.VersionBind, "| pypdf", pypdf.__version__)
free = shutil.disk_usage("/kaggle/working").free / 1e9
print(f"Free disk in /kaggle/working: {free:.1f} GB")

In [ ]:
# CELL 21 — Feature extractor (exact CELL 10 code) + PDF->graph converter (exact fixed CELL 12 code), path-based; quick self-test
import os, re, zlib, math, gzip, pickle, logging, collections
import numpy as np
import pandas as pd
import pikepdf

logging.getLogger("pikepdf").setLevel(logging.ERROR)

FEAT_TIME_LIMIT = 60
GRAPH_TIME_LIMIT = 90
MAX_STREAMS = 120
TOOL_WORDS = [b"pypdf", b"pymupdf", b"mupdf", b"pikepdf", b"qpdf"]
STREAM_BODY = re.compile(rb"(?<!end)stream\r?\n.*?endstream", re.S)
STD14 = {"Helvetica", "Helvetica-Bold", "Helvetica-Oblique", "Helvetica-BoldOblique", "Times-Roman", "Times-Bold",
         "Times-Italic", "Times-BoldItalic", "Courier", "Courier-Bold", "Courier-Oblique", "Courier-BoldOblique",
         "Symbol", "ZapfDingbats"}

def ratio(a, b):
    return a / b if b else 0.0

# =====================================================================================
# PART A — document-level features (identical to CELL 10)
# =====================================================================================
def file_features(raw):
    f = {}
    skel = STREAM_BODY.sub(b"stream endstream", raw)
    f["size_log"] = math.log10(len(raw) + 1)
    m = re.match(rb"\s*%PDF-(\d)\.(\d)", raw[:64])
    f["version"] = int(m.group(1)) + int(m.group(2)) / 10 if m else 0
    f["binary_comment"] = int(bool(re.match(rb"\s*%PDF-\d\.\d\s*[\r\n]+%[\x80-\xff]{4}", raw[:40])))
    f["n_eof"] = raw.count(b"%%EOF")
    f["n_startxref"] = raw.count(b"startxref")
    f["xref_table"] = int(bool(re.search(rb"(^|[\r\n])xref\s*[\r\n]", skel)))
    f["xref_stream"] = int(bool(re.search(rb"/Type\s*/XRef", skel)))
    f["obj_streams"] = int(bool(re.search(rb"/Type\s*/ObjStm", skel)))
    f["linearized"] = int(b"/Linearized" in raw[:2048])
    f["true_revisions"] = f["n_eof"] - f["linearized"]
    crlf = skel.count(b"\r\n")
    cr = skel.count(b"\r") - crlf
    lf = skel.count(b"\n") - crlf
    tot = crlf + cr + lf
    f["eol_crlf"], f["eol_cr"], f["eol_lf"] = ratio(crlf, tot), ratio(cr, tot), ratio(lf, tot)
    objs = re.findall(rb"(\d+)\s+(\d+)\s+obj", skel)
    nums = [int(a) for a, _ in objs]
    f["n_obj_defs"] = len(nums)
    f["obj_order_increasing"] = ratio(sum(1 for a, b in zip(nums, nums[1:]) if b > a), max(len(nums) - 1, 1))
    f["obj_gen_nonzero"] = ratio(sum(1 for _, g in objs if g != b"0"), len(objs))
    f["obj_max_over_count"] = ratio(max(nums) if nums else 0, len(set(nums)))
    f["obj_redefined"] = len(nums) - len(set(nums))
    f["obj_newline_dict"] = ratio(len(re.findall(rb"obj\s*[\r\n]+\s*<<", skel)), len(nums))
    f["dict_open_space"] = ratio(skel.count(b"<< /"), skel.count(b"<<"))
    f["dict_close_space"] = ratio(skel.count(b" >>"), skel.count(b">>"))
    nstream = len(re.findall(rb"(?<!end)stream", skel))
    f["stream_kw_crlf"] = ratio(len(re.findall(rb"(?<!end)stream\r\n", raw)), nstream)
    f["endobj_newline_before"] = ratio(len(re.findall(rb"[\r\n]endobj", skel)), skel.count(b"endobj"))
    f["reals_dot_zero"] = ratio(len(re.findall(rb"\s\d+\.0[\s/\]>]", skel)), len(re.findall(rb"\s\d+\.\d+[\s/\]>]", skel)))
    f["has_ID"] = int(b"/ID" in skel)
    f["tool_word_in_file"] = int(any(w in skel.lower() for w in TOOL_WORDS))
    return f

def stream_sig(s):
    d = {}
    flt = s.get("/Filter")
    names = [str(x) for x in flt] if isinstance(flt, pikepdf.Array) else ([] if flt is None else [str(flt)])
    d["filter"] = "+".join(names) if names else "none"
    d["has_parms"] = int(s.get("/DecodeParms") is not None)
    d["len_indirect"] = int(bool(getattr(s.get("/Length"), "is_indirect", False)))
    d["hdr"], d["zlevel"], d["sync_end"], d["adler_ok"], d["tail_junk"], d["ratio"] = -1, 0, 0, 0, 0, 0.0
    raw = s.read_raw_bytes()
    if names == ["/FlateDecode"] and len(raw) >= 6:
        d["hdr"] = raw[0] * 256 + raw[1]
        try:
            dob = zlib.decompressobj()
            dec = dob.decompress(raw)
            body = raw[:len(raw) - len(dob.unused_data)]
            d["tail_junk"] = int(len(dob.unused_data) > 0)
            d["ratio"] = ratio(len(body), len(dec))
            d["adler_ok"] = int(body[-4:] == zlib.adler32(dec).to_bytes(4, "big"))
            d["sync_end"] = int(b"\x00\x00\xff\xff" in body[-12:])
            if len(dec) <= 2_000_000:
                for lv in range(1, 10):
                    if zlib.compress(dec, lv) == body:
                        d["zlevel"] = lv
                        break
        except Exception:
            d["hdr"] = -2
    return d

def resource_features(pdf):
    f = {"l5_n_annots": 0, "l5_n_freetext": 0, "l5_n_text_notes": 0, "l5_n_img_xobj": 0, "l5_n_fonts": 0,
         "l5_n_std14_unembedded": 0, "l5_frac_pages_multi_stream": 0.0, "l5_max_streams_per_page": 0}
    fonts_seen, imgs_seen, multi = set(), set(), 0
    for page in pdf.pages:
        po = page.obj
        annots = po.get("/Annots")
        if isinstance(annots, pikepdf.Array):
            for a in annots:
                try:
                    st = str(a.get("/Subtype", ""))
                except Exception:
                    continue
                f["l5_n_annots"] += 1
                f["l5_n_freetext"] += int(st == "/FreeText")
                f["l5_n_text_notes"] += int(st == "/Text")
        c = po.get("/Contents")
        ns = len(c) if isinstance(c, pikepdf.Array) else (1 if c is not None else 0)
        multi += int(ns > 1)
        f["l5_max_streams_per_page"] = max(f["l5_max_streams_per_page"], ns)
        try:
            res = page.resources
        except Exception:
            res = po.get("/Resources")
        if res is None:
            continue
        fdict = res.get("/Font")
        if isinstance(fdict, pikepdf.Dictionary):
            for _, fo in fdict.items():
                key = getattr(fo, "objgen", (0, 0))
                key = key if key != (0, 0) else id(fo)
                if key in fonts_seen:
                    continue
                fonts_seen.add(key)
                base = str(fo.get("/BaseFont", "")).lstrip("/")
                desc = fo.get("/FontDescriptor")
                embedded = desc is not None and any(k in desc for k in ("/FontFile", "/FontFile2", "/FontFile3"))
                if base in STD14 and not embedded and str(fo.get("/Subtype", "")) != "/Type0":
                    f["l5_n_std14_unembedded"] += 1
        xdict = res.get("/XObject")
        if isinstance(xdict, pikepdf.Dictionary):
            for _, xo in xdict.items():
                key = getattr(xo, "objgen", (0, 0))
                if key in imgs_seen:
                    continue
                try:
                    if str(xo.get("/Subtype", "")) == "/Image":
                        imgs_seen.add(key)
                except Exception:
                    pass
    f["l5_n_fonts"] = len(fonts_seen)
    f["l5_n_img_xobj"] = len(imgs_seen)
    f["l5_frac_pages_multi_stream"] = ratio(multi, len(pdf.pages))
    return f

def extract_features(path):
    with open(path, "rb") as fh:
        raw = fh.read()
    f = {"out_file": os.path.basename(path)}
    f.update(file_features(raw))
    with pikepdf.open(path) as pdf:
        info = pdf.docinfo
        prod = str(info.get("/Producer", "")) if info is not None else ""
        f["has_producer"] = int(bool(prod.strip()))
        f["producer_len"] = len(prod)
        cd, md = str(info.get("/CreationDate", "")), str(info.get("/ModDate", ""))
        f["moddate_eq_creation"] = int(bool(cd) and cd == md)
        f["has_xmp"] = int("/Metadata" in pdf.Root)
        f["n_pages"] = len(pdf.pages)
        try:
            f.update(resource_features(pdf))
        except Exception:
            pass
        streams = []
        for pi, page in enumerate(pdf.pages):
            c = page.obj.get("/Contents")
            if c is None:
                continue
            for s in (list(c) if isinstance(c, pikepdf.Array) else [c]):
                if isinstance(s, pikepdf.Stream):
                    streams.append((pi, s))
            if len(streams) >= MAX_STREAMS:
                break
        streams = streams[:MAX_STREAMS]
        sigs = []
        for pi, s in streams:
            try:
                sigs.append((pi, getattr(s, "objgen", (-1, 0))[0], stream_sig(s)))
            except Exception:
                pass
        cs_nums = {num for _, num, _ in sigs}

    n = len(sigs)
    f["cs_n"] = n
    if n:
        fl = [d["filter"] for _, _, d in sigs]
        f["cs_frac_flate"] = ratio(fl.count("/FlateDecode"), n)
        f["cs_frac_lzw"] = ratio(sum("LZW" in x for x in fl), n)
        f["cs_frac_none"] = ratio(fl.count("none"), n)
        f["cs_n_filters"] = len(set(fl))
        flate = [d for _, _, d in sigs if d["filter"] == "/FlateDecode"]
        f["cs_frac_zlib_repro"] = ratio(sum(d["zlevel"] > 0 for d in flate), len(flate)) if flate else np.nan
        f["cs_n_hdr"] = len({d["hdr"] for _, _, d in sigs})
        f["cs_n_zlevel"] = len({d["zlevel"] for _, _, d in sigs})
        f["cs_frac_sync"] = np.mean([d["sync_end"] for _, _, d in sigs])
        f["cs_frac_adler"] = np.mean([d["adler_ok"] for _, _, d in sigs])
        f["cs_frac_junk"] = np.mean([d["tail_junk"] for _, _, d in sigs])
        f["cs_frac_len_indirect"] = np.mean([d["len_indirect"] for _, _, d in sigs])
        r = np.array([d["ratio"] for _, _, d in sigs])
        f["cs_ratio_std"] = float(r.std())
        f["cs_ratio_max_z"] = float(np.max(np.abs(r - r.mean())) / r.std()) if r.std() > 0 else 0.0
        keys = [(d["filter"], d["hdr"], d["zlevel"], d["sync_end"], d["adler_ok"], d["tail_junk"], d["has_parms"], d["len_indirect"]) for _, _, d in sigs]
        cnt = collections.Counter(keys)
        f["cs_n_sig"] = len(cnt)
        f["cs_majority_share"] = max(cnt.values()) / n
        f["cs_n_singletons"] = sum(1 for v in cnt.values() if v == 1) if n >= 3 else 0
        odd = [pi for (pi, _, _), k in zip(sigs, keys) if cnt[k] == 1] if n >= 3 else []
        f["loc_page"] = odd[0] if len(odd) == 1 else -1

    f.update({"upd_n_objs": 0, "upd_touch_content": 0, "upd_has_annot": 0, "upd_has_page": 0, "upd_has_sig": 0})
    if f["true_revisions"] > 1:
        ends = [m.end() for m in re.finditer(rb"%%EOF", raw)]
        tail = STREAM_BODY.sub(b"stream endstream", raw[ends[-2]:])
        objs = re.findall(rb"(\d+)\s+\d+\s+obj(.*?)endobj", tail, re.S)
        f["upd_n_objs"] = len(objs)
        for num, body in objs:
            if int(num) in cs_nums:
                f["upd_touch_content"] = 1
            if re.search(rb"/Type\s*/Annot|/Subtype\s*/(Text|FreeText)\b", body):
                f["upd_has_annot"] = 1
            if re.search(rb"/Type\s*/Page\b", body):
                f["upd_has_page"] = 1
            if b"/ByteRange" in body:
                f["upd_has_sig"] = 1
    return f

# =====================================================================================
# PART B — PDF -> graph (identical to fixed CELL 12)
# =====================================================================================
NODE_TYPES = ["catalog", "pages", "page", "content", "font", "fontdesc", "fontfile", "image", "form", "annot",
              "metadata", "xref", "extgstate", "other_dict", "array", "other_stream", "other"]
EDGE_KEYS = ["/Parent", "/Kids", "/Contents", "/Resources", "/Font", "/XObject", "/Annots", "/FontDescriptor",
             "/FontFile", "/Metadata", "/Pages", "other"]
HDRS = [0x789C, 0x78DA, 0x7801, 0x785E]
FEAT_NAMES = (["t_" + t for t in NODE_TYPES] + ["is_stream", "raw_len_log", "n_keys_log"] +
              ["f_flate", "f_lzw", "f_dct", "f_none", "f_other"] +
              ["h_789C", "h_78DA", "h_7801", "h_785E", "h_other"] +
              ["zlevel", "zlib_repro", "adler_ok", "sync_end", "tail_junk", "ratio",
               "objnum_rank", "gen_nonzero", "std14_unembedded", "annot_freetext", "annot_text",
               "len_indirect", "entropy"])
D = len(FEAT_NAMES)
LEN_REF = re.compile(rb"/Length\s+\d+\s+\d+\s+R")

def edge_key(k):
    if k is None:
        return "other"
    if k.startswith("/FontFile"):
        return "/FontFile"
    return k if k in EDGE_KEYS else "other"

def collect_refs(o, carried, out, depth=0):
    if depth > 8:
        return
    try:
        if isinstance(o, (pikepdf.Dictionary, pikepdf.Stream)):
            for k, v in o.items():
                kk = k if (k in EDGE_KEYS or k.startswith("/FontFile")) else carried
                if getattr(v, "is_indirect", False):
                    out.append((edge_key(kk), v.objgen))
                else:
                    collect_refs(v, kk, out, depth + 1)
        elif isinstance(o, pikepdf.Array):
            for v in o:
                if getattr(v, "is_indirect", False):
                    out.append((edge_key(carried), v.objgen))
                else:
                    collect_refs(v, carried, out, depth + 1)
    except Exception:
        pass

def node_type(o, content_set):
    if isinstance(o, pikepdf.Stream):
        if o.objgen in content_set:
            return "content"
        st, ty = str(o.get("/Subtype", "")), str(o.get("/Type", ""))
        if st == "/Image":
            return "image"
        if st == "/Form":
            return "form"
        if ty == "/Metadata":
            return "metadata"
        if ty in ("/XRef", "/ObjStm"):
            return "xref"
        if "/Length1" in o or st in ("/Type1C", "/CIDFontType0C", "/OpenType"):
            return "fontfile"
        return "other_stream"
    if isinstance(o, pikepdf.Dictionary):
        ty, st = str(o.get("/Type", "")), str(o.get("/Subtype", ""))
        mapping = {"/Catalog": "catalog", "/Pages": "pages", "/Page": "page", "/Font": "font",
                   "/FontDescriptor": "fontdesc", "/Annot": "annot", "/ExtGState": "extgstate"}
        if ty in mapping:
            return mapping[ty]
        if st in ("/Text", "/FreeText", "/Link", "/Widget", "/Stamp", "/Square", "/Highlight"):
            return "annot"
        return "other_dict"
    if isinstance(o, pikepdf.Array):
        return "array"
    return "other"

def shannon(raw):
    if not raw:
        return 0.0
    b = np.frombuffer(raw[:2_000_000], dtype=np.uint8)
    c = np.bincount(b, minlength=256)
    p = c[c > 0] / len(b)
    return float(-(p * np.log2(p)).sum() / 8)

def stream_feats(o, is_content):
    v = {"raw_len_log": 0.0, "f": "f_none", "h": None, "zlevel": 0, "zlib_repro": 0, "adler_ok": 0,
         "sync_end": 0, "tail_junk": 0, "ratio": 0.0, "len_indirect": 0, "entropy": 0.0}
    try:
        v["len_indirect"] = int(bool(LEN_REF.search(o.stream_dict.unparse())))
    except Exception:
        pass
    flt = o.get("/Filter")
    names = [str(x) for x in flt] if isinstance(flt, pikepdf.Array) else ([] if flt is None else [str(flt)])
    raw = o.read_raw_bytes()
    v["raw_len_log"] = math.log10(len(raw) + 1) / 7
    v["entropy"] = shannon(raw)
    if not names:
        v["f"] = "f_none"
    elif names == ["/FlateDecode"]:
        v["f"] = "f_flate"
    elif any("LZW" in n for n in names):
        v["f"] = "f_lzw"
    elif any("DCT" in n for n in names):
        v["f"] = "f_dct"
    else:
        v["f"] = "f_other"
    if v["f"] == "f_flate" and len(raw) >= 6:
        hdr = raw[0] * 256 + raw[1]
        v["h"] = "h_%04X" % hdr if hdr in HDRS else "h_other"
        if is_content or len(raw) <= 200_000:
            try:
                dob = zlib.decompressobj()
                dec = dob.decompress(raw)
                body = raw[:len(raw) - len(dob.unused_data)]
                v["tail_junk"] = int(len(dob.unused_data) > 0)
                v["ratio"] = min(len(body) / max(len(dec), 1), 1.5)
                v["adler_ok"] = int(body[-4:] == zlib.adler32(dec).to_bytes(4, "big"))
                v["sync_end"] = int(b"\x00\x00\xff\xff" in body[-12:])
                if len(dec) <= 2_000_000:
                    for lv in range(1, 10):
                        if zlib.compress(dec, lv) == body:
                            v["zlevel"], v["zlib_repro"] = lv, 1
                            break
            except Exception:
                pass
    return v

def to_graph(path):
    with pikepdf.open(path) as pdf:
        content_page = {}
        for pi, page in enumerate(pdf.pages):
            c = page.obj.get("/Contents")
            if c is None:
                continue
            for s in (list(c) if isinstance(c, pikepdf.Array) else [c]):
                if isinstance(s, pikepdf.Stream) and s.objgen not in content_page:
                    content_page[s.objgen] = pi
        objs = [o for o in pdf.objects if hasattr(o, "objgen") and o.objgen != (0, 0)]
        idx = {o.objgen: i for i, o in enumerate(objs)}
        max_num = max([g[0] for g in idx] + [1])
        X = np.zeros((len(objs), D), dtype=np.float16)
        node_page = np.full(len(objs), -1, dtype=np.int16)
        col = {n: j for j, n in enumerate(FEAT_NAMES)}
        src, dst, et = [], [], []
        for i, o in enumerate(objs):
            t = node_type(o, content_page)
            X[i, col["t_" + t]] = 1
            X[i, col["objnum_rank"]] = o.objgen[0] / max_num
            X[i, col["gen_nonzero"]] = int(o.objgen[1] != 0)
            if o.objgen in content_page:
                node_page[i] = content_page[o.objgen]
            try:
                if isinstance(o, (pikepdf.Dictionary, pikepdf.Stream)):
                    X[i, col["n_keys_log"]] = math.log10(len(list(o.keys())) + 1)
                if isinstance(o, pikepdf.Stream):
                    X[i, col["is_stream"]] = 1
                    sf = stream_feats(o, t == "content")
                    X[i, col["raw_len_log"]] = sf["raw_len_log"]
                    X[i, col[sf["f"]]] = 1
                    if sf["h"]:
                        X[i, col[sf["h"]]] = 1
                    X[i, col["zlevel"]] = sf["zlevel"] / 9
                    for k in ["zlib_repro", "adler_ok", "sync_end", "tail_junk", "ratio", "len_indirect", "entropy"]:
                        X[i, col[k]] = sf[k]
                if t == "font":
                    base = str(o.get("/BaseFont", "")).lstrip("/")
                    desc = o.get("/FontDescriptor")
                    emb = desc is not None and any(k in desc for k in ("/FontFile", "/FontFile2", "/FontFile3"))
                    X[i, col["std14_unembedded"]] = int(base in STD14 and not emb and str(o.get("/Subtype", "")) != "/Type0")
                if t == "annot":
                    st = str(o.get("/Subtype", ""))
                    X[i, col["annot_freetext"]] = int(st == "/FreeText")
                    X[i, col["annot_text"]] = int(st == "/Text")
            except Exception:
                pass
            refs = []
            collect_refs(o, None, refs)
            for k, tgt in refs:
                if tgt in idx:
                    src.append(i)
                    dst.append(idx[tgt])
                    et.append(EDGE_KEYS.index(k))
    return {"out_file": os.path.basename(path), "x": X, "edge_index": np.array([src, dst], dtype=np.int32),
            "edge_type": np.array(et, dtype=np.int8), "node_page": node_page}

# =====================================================================================
# Quick self-test on 3 original PDFs (reads only, writes nothing)
# =====================================================================================
for name, fam in [JOBS[0], JOBS[len(JOBS) // 2], JOBS[-1]]:
    p = os.path.join(SRC, name)
    fe = extract_features(p)
    gr = to_graph(p)
    print(f"{name} ({fam}): {len(fe) - 1} feature values | graph {gr['x'].shape[0]} nodes, "
          f"{gr['edge_index'].shape[1]} edges, {gr['x'].shape[1]} node features")
print("\nCELL 21 ready. Expected: 62 feature values (61 if a file has no page streams), 43 node features.")

In [ ]:
# CELL 41 (v2) — ONE clean setup for the baseline tools (replaces CELLS 34-40). Checks every tool WORKS (not just exists), rebuilds anything broken, then replaces symlinks with real copies so Kaggle "Files only" persistence keeps them. Safe to rerun.
import os, sys, glob, json, shutil, subprocess

assert "SRC" in globals() and "extract_features" in globals(), "Run CELL 20 and CELL 21 first"

B, EXT, TEST = "/kaggle/working/baselines", "/kaggle/working/baselines/ext", "/kaggle/working/baselines/test_pdfs"
for d in (B, EXT, TEST):
    os.makedirs(d, exist_ok=True)

def sh(cmd, timeout=600, cwd=None):
    """Run a shell command. Never raises. Returns (return code, combined output)."""
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, cwd=cwd)
        return r.returncode, (r.stdout or "") + (r.stderr or "")
    except subprocess.TimeoutExpired:
        return -9, f"TIMEOUT after {timeout}s"

def tail(s, n=25):
    return "\n".join(s.strip().splitlines()[-n:])

S = {}
def step(name, ok, note=""):
    S[name] = bool(ok)
    print(f"[{'✅' if ok else '❌'}] {name}" + (f" | {note}" if note else ""))

def materialize(root):
    """Replace every symlink under root with a real copy of its target (Kaggle persistence drops symlinks)."""
    n, broken = 0, 0
    for dirpath, dirnames, filenames in os.walk(root):
        for name in dirnames + filenames:
            p = os.path.join(dirpath, name)
            if os.path.islink(p):
                t = os.path.realpath(p)
                if not os.path.exists(t):
                    broken += 1
                    continue
                os.unlink(p)
                if os.path.isdir(t):
                    shutil.copytree(t, p, symlinks=False)
                else:
                    shutil.copy2(t, p)
                n += 1
    return n, broken

P, psrc, ptar = f"{EXT}/poppler057", f"{EXT}/poppler-0.57.0", f"{EXT}/poppler-0.57.0.tar.xz"
MM_DIR, MM_ROOT, PY27 = f"{EXT}/mm", f"{EXT}/mmroot", f"{EXT}/py27"
mm, py27 = f"{MM_DIR}/bin/micromamba", f"{PY27}/bin/python"

# ---------- diagnosis (what survived the restart) ----------
plibs = sorted(os.path.basename(p) for p in glob.glob(f"{P}/lib*/libpoppler.so*"))
print(f"   diagnosis | poppler057 lib files: {plibs} | py27/bin/python present: {os.path.exists(py27)}"
      f" | py27/bin/python2.7 present: {os.path.exists(f'{PY27}/bin/python2.7')}")

# ---------- a. internet ----------
rc, out = sh("curl -sI -m 20 https://github.com | head -1", timeout=40)
step("a. Internet on Kaggle", rc == 0 and "HTTP" in out)

# ---------- b. system libraries (same packages that worked in CELLS 34-39) ----------
APT = ("cmake g++ make pkg-config git curl xz-utils bzip2 "
       "libboost-dev libboost-filesystem-dev libboost-program-options-dev libboost-regex-dev "
       "libboost-system-dev libboost-thread-dev default-jdk-headless "
       "libjpeg-dev libfontconfig1-dev libfreetype6-dev zlib1g-dev")
rc, out = sh("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq " + APT, timeout=1800)
step("b. System libraries", rc == 0, "" if rc == 0 else tail(out, 4))
if os.path.exists("/usr/include/poppler/PDFDoc.h"):
    sh("DEBIAN_FRONTEND=noninteractive apt-get remove -y -qq libpoppler-dev libpoppler-private-dev", timeout=600)
step("b. No system Poppler headers (avoids clash)", not os.path.exists("/usr/include/poppler/PDFDoc.h"))

# ---------- c. libquickly (installs into /usr/local, which is lost at every restart) ----------
lq = f"{EXT}/libquickly"
if not os.path.isdir(lq):
    sh(f"git clone -q --depth 1 https://github.com/srndic/libquickly.git {lq}", timeout=300)
shutil.rmtree(f"{lq}/build", ignore_errors=True)
os.makedirs(f"{lq}/build")
rc, out = sh("cmake .. && make -j4 && make install && ldconfig", cwd=f"{lq}/build", timeout=900)
step("c. libquickly built + installed", rc == 0, "" if rc == 0 else tail(out, 6))

# ---------- d. Poppler 0.57.0: must have a WORKING libpoppler.so + headers, else clean rebuild ----------
def poppler_libdir():
    for d in sorted(glob.glob(f"{P}/lib*")):
        if os.path.exists(f"{d}/libpoppler.so"):            # exists() follows links: False if link broken/missing
            return d
    return None

if poppler_libdir() is None or not os.path.exists(f"{P}/include/poppler/PDFDoc.h"):
    print("   Poppler 0.57.0 missing or broken -> clean rebuild (~10-20 min) ...")
    shutil.rmtree(P, ignore_errors=True)
    shutil.rmtree(psrc, ignore_errors=True)                 # fresh source (old one may have lost symlinks too)
    if not os.path.exists(ptar):
        sh(f"curl -fL -sS -m 600 -o {ptar} https://poppler.freedesktop.org/poppler-0.57.0.tar.xz", timeout=700)
    sh(f"tar -xf {ptar} -C {EXT}", timeout=300)
    pb = f"{psrc}/build"
    os.makedirs(pb, exist_ok=True)
    opts = " ".join([
        f"-DCMAKE_INSTALL_PREFIX={P}", "-DCMAKE_BUILD_TYPE=Release", "-DENABLE_XPDF_HEADERS=ON",
        "-DENABLE_DCTDECODER=libjpeg", "-DENABLE_LIBOPENJPEG=none", '-DENABLE_CMS=""',
        "-DENABLE_SPLASH=OFF", "-DENABLE_UTILS=OFF", "-DENABLE_CPP=OFF", "-DENABLE_LIBCURL=OFF",
        "-DBUILD_GTK_TESTS=OFF", "-DBUILD_QT4_TESTS=OFF", "-DBUILD_QT5_TESTS=OFF", "-DBUILD_CPP_TESTS=OFF",
        "-DCMAKE_DISABLE_FIND_PACKAGE_GLIB=ON", "-DCMAKE_DISABLE_FIND_PACKAGE_Cairo=ON",
        "-DCMAKE_DISABLE_FIND_PACKAGE_Qt4=ON", "-DCMAKE_DISABLE_FIND_PACKAGE_Qt5Core=ON",
        "-DCMAKE_DISABLE_FIND_PACKAGE_GTK=ON", "-DCMAKE_POLICY_VERSION_MINIMUM=3.5",
        '-DCMAKE_CXX_FLAGS="-std=c++11 -fpermissive -w"', '-DCMAKE_C_FLAGS="-w"'])
    rc, out = sh(f"cmake .. {opts} && make -j4 poppler && make install", cwd=pb, timeout=3000)
    if rc != 0:
        print(tail(out, 8))
libdir = poppler_libdir()
if libdir:
    n, br = materialize(P)
    print(f"   Poppler: {n} symlinks replaced by real files ({br} broken)")
libdir = poppler_libdir()
step("d. Poppler 0.57.0 (libpoppler.so + PDFDoc.h)", libdir is not None and os.path.exists(f"{P}/include/poppler/PDFDoc.h"), str(libdir))

if libdir:
    os.environ["CPLUS_INCLUDE_PATH"] = f"{P}/include/poppler:{P}/include"
    os.environ["LIBRARY_PATH"] = libdir
    if libdir not in os.environ.get("LD_LIBRARY_PATH", ""):
        os.environ["LD_LIBRARY_PATH"] = libdir + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# ---------- e. Hidost PDF tools (original code, unchanged; rebuilt if missing or broken) ----------
PDF_TOOLS = {"cacher", "pdf2paths", "pdf2vals", "pathcount", "feat-select", "feat-extract", "merger"}
h57 = f"{EXT}/hidost_p057"

def find_tools():
    found = {}
    for root, _, files in os.walk(f"{h57}/build"):
        for fn in files:
            fp = os.path.join(root, fn)
            if fn in PDF_TOOLS and os.access(fp, os.X_OK):
                found[fn] = fp
    return found

def missing_libs(exe):
    _, o = sh(f'ldd "{exe}"', timeout=30)
    return [l.split()[0] for l in o.splitlines() if "not found" in l]

HIDOST = find_tools()
if libdir and (set(HIDOST) != PDF_TOOLS or any(missing_libs(x) for x in HIDOST.values())):
    print("   (re)building Hidost PDF tools ...")
    if not os.path.isdir(f"{EXT}/hidost"):
        sh(f"git clone -q --depth 1 https://github.com/srndic/hidost.git {EXT}/hidost", timeout=300)
    shutil.rmtree(h57, ignore_errors=True)
    shutil.copytree(f"{EXT}/hidost", h57, ignore=shutil.ignore_patterns("build"))
    os.makedirs(f"{h57}/build")
    rc, out = sh(f"cmake .. -DCMAKE_LIBRARY_PATH={libdir} -DCMAKE_POLICY_VERSION_MINIMUM=3.5", cwd=f"{h57}/build", timeout=300)
    if rc == 0:
        _, hl = sh("make help", cwd=f"{h57}/build", timeout=60)
        want = [l.split("... ", 1)[1].split()[0] for l in hl.splitlines()
                if l.startswith("... ") and l.split("... ", 1)[1].split()[0] in PDF_TOOLS]
        rc, out = sh(f"make -k -j4 {' '.join(want)}", cwd=f"{h57}/build", timeout=1800)
    if rc != 0:
        print(tail(out, 8))
    HIDOST = find_tools()
bad = {}
for name, exe in sorted(HIDOST.items()):
    miss = missing_libs(exe)
    _, o = sh(f'"{exe}" --help', timeout=20)
    if miss or "error while loading shared libraries" in o:
        bad[name] = miss or [tail(o, 1)]
step("e. Hidost: all 7 PDF tools present", set(HIDOST) == PDF_TOOLS, str(sorted(HIDOST)))
step("e. Hidost: every tool finds its libraries and starts", len(HIDOST) == 7 and not bad, str(bad) if bad else "")

# ---------- f. micromamba + Python 2.7: must actually RUN, else clean recreate ----------
os.environ["MAMBA_ROOT_PREFIX"] = MM_ROOT

def py27_check():
    rc, v = sh(f'"{py27}" -c "import sys, numpy, sklearn, dateutil; print(sys.version.split()[0], numpy.__version__, sklearn.__version__)"',
               timeout=120)
    return rc == 0 and "'2.7." in v, v.strip()[:60]

ok27, v27 = py27_check()
if not ok27:
    print("   Python 2.7 env missing or broken -> clean recreate (~5 min) ...")
    _, mv = sh(f'"{mm}" --version', timeout=30)
    if not mv.strip()[:1].isdigit():
        shutil.rmtree(MM_DIR, ignore_errors=True)
        os.makedirs(MM_DIR, exist_ok=True)
        sh(f"curl -fLs -m 300 https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C {MM_DIR} bin/micromamba", timeout=360)
    shutil.rmtree(PY27, ignore_errors=True)
    shutil.rmtree(MM_ROOT, ignore_errors=True)             # package cache may also have lost symlinks
    rc, out = sh(f'"{mm}" create -y -p {PY27} -c https://repo.anaconda.com/pkgs/main --override-channels '
                 "python=2.7 numpy scikit-learn python-dateutil", timeout=2400)
    if rc != 0:
        print(tail(out, 6))
    if os.path.isdir(PY27):
        n, br = materialize(PY27)
        print(f"   Python 2.7 env: {n} symlinks replaced by real files ({br} broken)")
    ok27, v27 = py27_check()
step("f. Python 2.7 env (python, numpy, scikit-learn)", ok27, v27)
if not os.path.isdir(f"{EXT}/mimicus"):
    sh(f"git clone -q --depth 1 https://github.com/srndic/mimicus.git {EXT}/mimicus", timeout=300)
step("f. Original Mimicus present", os.path.exists(f"{EXT}/mimicus/mimicus/tools/featureedit.py"))

# ---------- g. Mimicus on one real PDF: must give 135 real numbers ----------
runner = f"{B}/run_pdfrate_py27.py"                       # identical to CELL 40 (v2)
with open(runner, "w") as fh:
    fh.write('''from __future__ import print_function
import sys, json, traceback, numpy
sys.path.insert(0, "MIMICUS_ROOT")
from mimicus.tools.featureedit import FeatureEdit, FeatureDescriptor
res = {"names": list(FeatureDescriptor.get_feature_names()), "files": {}}
for p in sys.argv[1:]:
    try:
        fe = FeatureEdit(p)
        d = fe.retrieve_feature_dictionary()
        size_bad = d["size"] == 0 or isinstance(d["size"], Exception)
        v = numpy.asarray(fe.retrieve_feature_vector_numpy())
        res["files"][p] = {"ok": True, "rejected_by_mimicus": bool(size_bad), "raw_shape": list(v.shape),
                           "vec": [float(x) for x in v.ravel()]}
    except Exception:
        res["files"][p] = {"ok": False, "err": traceback.format_exc()[-600:]}
print("JSON_START" + json.dumps(res))
'''.replace("MIMICUS_ROOT", f"{EXT}/mimicus"))
probe = f"{SRC}/000010.pdf" if os.path.exists(f"{SRC}/000010.pdf") else sorted(glob.glob(f"{SRC}/*.pdf"))[0]
rc, out = sh(f'"{py27}" "{runner}" "{probe}"', timeout=300)
ok_g, note = False, tail(out, 3)
if "JSON_START" in out:
    r = json.loads(out.split("JSON_START", 1)[1].strip().splitlines()[0])
    x = r["files"].get(probe, {})
    if x.get("ok"):
        nz = sum(v != 0 for v in x["vec"])
        ok_g = len(r["names"]) == 135 and len(x["vec"]) == 135 and nz > 0 and not x["rejected_by_mimicus"]
        note = f"{os.path.basename(probe)}: {len(x['vec'])} numbers, {nz} non-zero"
    else:
        note = x.get("err", "")[-150:]
step("g. Mimicus gives 135 real numbers on a real PDF", ok_g, note)

# ---------- summary ----------
BASELINE_TOOLS = {"hidost": HIDOST, "poppler_lib": libdir, "py27": py27, "mimicus": f"{EXT}/mimicus", "pdfrate_runner": runner}
with open(f"{B}/setup_report.json", "w") as fh:
    json.dump({"status": S, "tools": BASELINE_TOOLS}, fh, indent=2)
print("\n=== CELL 41 SUMMARY ===")
print("ALL READY ✅" if all(S.values()) else "NOT READY ❌ -> failed: " + ", ".join(k for k, v in S.items() if not v))
print(f"Report saved: {B}/setup_report.json")

In [ ]:
# CELL 44 — Big baseline extraction pass. Per chunk: regenerate the same 15 versions (CELL 20 generator, same seeds) for the documents in full_features.csv -> PDFrate (original Mimicus, Python 2.7) -> Hidost cacher (authors' options, folders pre-created, 1 retry) -> save -> delete PDFs. Resumable. TRIAL=True: 20 documents in a separate folder, with timing + checks.
import os, glob, json, time, shutil, stat, subprocess, tarfile
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, TimeoutError, as_completed
from pebble import ProcessPool
from tqdm.auto import tqdm

TRIAL = False     # True = 20 documents (speed + correctness check). Set False for the full run (Save & Run All).

assert "process" in globals() and "JOBS" in globals(), "Run CELL 20 and CELL 21 first"
assert "HIDOST" in globals() and len(HIDOST) == 7 and os.path.exists(py27), "Run CELL 41 first (it must say ALL READY)"
RUNNER = f"{B}/run_pdfrate_py27.py"                     # written by CELL 41 (identical to CELL 40 v2)
assert os.path.exists(RUNNER), "PDFrate runner missing - rerun CELL 41"

if not (os.path.exists("/dev/null") and stat.S_ISCHR(os.stat("/dev/null").st_mode)):
    subprocess.run("rm -f /dev/null && mknod -m 666 /dev/null c 1 3", shell=True)

BUILD = f"{EXT}/hidost_p057/build"
T = {k: "./" + os.path.relpath(v, BUILD) for k, v in HIDOST.items()}
BIG = f"{B}/big_trial" if TRIAL else f"{B}/big"
if TRIAL:
    shutil.rmtree(BIG, ignore_errors=True)
HC, CK = f"{BIG}/hidost_cache", f"{BIG}/chunks"
os.makedirs(HC, exist_ok=True)
os.makedirs(CK, exist_ok=True)

# ---------- reference: the exact files of the main dataset ----------
hf = sorted(glob.glob("/kaggle/input/**/full/full_features.csv", recursive=True))
assert hf, "full_features.csv not found - is BTP Full attached as input?"
REF = pd.read_csv(hf[0], usecols=["out_file", "orig_file", "family", "variant", "label", "split", "twin", "edit_page"])
print(f"Main dataset: {len(REF)} files | {REF.orig_file.nunique()} documents | "
      f"train docs {REF[REF.split == 'train'].orig_file.nunique()} | dev docs {REF[REF.split == 'dev'].orig_file.nunique()}")
KEEP = set(REF.orig_file)
jobs_all = [j for j in JOBS if j[0] in KEEP]
assert len(jobs_all) == len(KEEP), "some documents of full_features.csv are not in the job list"
if TRIAL:
    step = max(1, len(jobs_all) // 20)
    jobs_all = jobs_all[::step][:20]                      # spread across families (JOBS is sorted by family)
CHUNKS_B = [jobs_all[i:i + CHUNK_SIZE] for i in range(0, len(jobs_all), CHUNK_SIZE)]
ref_idx = REF.set_index("out_file")
print(f"Mode: {'TRIAL (20 documents)' if TRIAL else 'FULL'} | documents: {len(jobs_all)} | chunks: {len(CHUNKS_B)} | output: {BIG}")

def run_pool(fn, items, timeout, desc):                   # same as CELL 22
    results, errors = {}, {}
    with ProcessPool(max_workers=WORKERS) as pool_ex:
        futs = {pool_ex.schedule(fn, args=(it,), timeout=timeout): it for it in items}
        for fu in tqdm(as_completed(futs), total=len(futs), desc=desc):
            it = futs[fu]
            try:
                results[it] = fu.result()
            except TimeoutError:
                errors[it] = "Timeout"
            except Exception as ex:
                errors[it] = f"{type(ex).__name__}: {str(ex)[:60]}"
    return results, errors

def save_atomic_csv(df, path):
    df.to_csv(path + ".tmp", index=False)
    os.replace(path + ".tmp", path)

# ---------- PDFrate: original Mimicus in Python 2.7, batches of 20 files in parallel, single-file retry ----------
def mimicus(paths, timeout):
    rc, out = sh(f'"{py27}" "{RUNNER}" ' + " ".join(f'"{p}"' for p in paths), timeout=timeout)
    if "JSON_START" not in out:
        return None
    return json.loads(out.split("JSON_START", 1)[1].strip().splitlines()[0])

def pdfrate(paths, desc):
    vec, err, names, retry = {}, {}, None, []

    def take(p, x):
        if x.get("ok") and len(x["vec"]) == 135 and not x["rejected_by_mimicus"]:
            vec[p] = x["vec"]
        else:
            err[p] = ("rejected by Mimicus" if x.get("ok") else x.get("err", "no result").strip().splitlines()[-1])[:120]

    batches = [paths[i:i + 20] for i in range(0, len(paths), 20)]
    with ThreadPoolExecutor(WORKERS) as ex:
        for b, res in tqdm(ex.map(lambda b: (b, mimicus(b, 20 * 45)), batches), total=len(batches), desc=desc):
            if res is None:
                retry += b
                continue
            names = names or res["names"]
            for p in b:
                take(p, res["files"].get(p, {}))
    if retry:
        with ThreadPoolExecutor(WORKERS) as ex:
            for p, res in ex.map(lambda p: (p, mimicus([p], 120)), retry):
                if res is None:
                    err[p] = "timeout or crash"
                else:
                    names = names or res["names"]
                    take(p, res["files"].get(p, {}))
    return vec, err, names

# ---------- Hidost: authors' cacher, cache folders created first, 1 retry ----------
def hc_path(p):                                           # cacher.cpp: cache_dir / <canonical input path>
    return os.path.join(HC, os.path.realpath(p).lstrip("/"))

def hc_ok(p):
    q = hc_path(p)
    return os.path.exists(q) and os.path.getsize(q) > 0

def hidost_cache(paths, ci):
    for d in {os.path.dirname(hc_path(p)) for p in paths}:
        os.makedirs(d, exist_ok=True)
    todo = list(paths)
    for attempt in (1, 2):
        lst = f"{CK}/hidost_list_{ci:02d}.txt"
        with open(lst, "w") as fh:
            fh.write("".join(p + "\n" for p in todo))
        sh(f"{T['cacher']} -i {lst} --compact --values -c {HC}/ -t10 -m256", cwd=BUILD, timeout=7200)
        todo = [p for p in todo if not hc_ok(p)]
        if not todo:
            break
    return {p for p in paths if hc_ok(p)}

# ---------- one chunk ----------
TIMES = {"generate": 0.0, "pdfrate": 0.0, "hidost": 0.0, "files": 0, "docs": 0}

def run_chunk(ci):
    f_pr, f_st = f"{CK}/chunk_{ci:02d}_pdfrate.csv", f"{CK}/chunk_{ci:02d}_status.csv"
    if os.path.exists(f_pr) and os.path.exists(f_st):
        print(f"Chunk {ci}: already done, skipping")
        return
    jobs = CHUNKS_B[ci]
    for p in glob.glob(os.path.join(OUT, "*.pdf")):      # start clean
        os.remove(p)
    t0 = time.time()
    gen, gerr = run_pool(process, jobs, TIME_LIMIT, f"Chunk {ci} | 1/3 generate")
    TIMES["generate"] += time.time() - t0
    rows, fails = [], []
    for job in jobs:
        res = gen.get(job)
        if res is not None and res[1] is not None:
            rows += res[1]
        else:
            fails.append({"orig_file": job[0], "stage": "generate", "reason": gerr.get(job) or (res[2] if res else "missing")})
            cleanup(job[0][:-4])



    if fails:
        print("Generation failures:", pd.Series([str(x["reason"])[:100] for x in fails]).value_counts().head(5).to_dict())
    assert rows, "No document could be generated - see 'Generation failures' above and send it to Claude"
    man = pd.DataFrame(rows)
    man = man[man.out_file.isin(ref_idx.index)].reset_index(drop=True)      # only files of the main dataset
    mism = int(sum((ref_idx.loc[r.out_file, "variant"] != r.variant) or (int(ref_idx.loc[r.out_file, "label"]) != int(r.label))
                   or (ref_idx.loc[r.out_file, "split"] != r.split) for r in man.itertuples()))
    assert mism == 0, f"{mism} regenerated files disagree with full_features.csv (variant/label/split) - send this to Claude"
    paths = [os.path.join(OUT, f) for f in man.out_file]

    t0 = time.time()
    vec, perr, names = pdfrate(paths, f"Chunk {ci} | 2/3 PDFrate")
    TIMES["pdfrate"] += time.time() - t0
    t0 = time.time()
    hok = hidost_cache(paths, ci)
    TIMES["hidost"] += time.time() - t0
    TIMES["files"] += len(paths)
    TIMES["docs"] += man.orig_file.nunique()

    assert names is not None and len(names) == 135, "Mimicus feature names not received"
    pr = pd.DataFrame([[os.path.basename(p)] + vec[p] for p in paths if p in vec], columns=["out_file"] + names)
    st = pd.DataFrame({"out_file": [os.path.basename(p) for p in paths],
                       "pdfrate_ok": [p in vec for p in paths],
                       "pdfrate_err": [perr.get(p, "") for p in paths],
                       "hidost_ok": [p in hok for p in paths],
                       "hidost_cache": [os.path.relpath(hc_path(p), HC) for p in paths]})
    if fails:
        st = pd.concat([st, pd.DataFrame([{"out_file": "", "pdfrate_ok": False, "pdfrate_err": f"{x['orig_file']} {x['stage']}: {x['reason']}",
                                           "hidost_ok": False, "hidost_cache": ""} for x in fails])], ignore_index=True)
    save_atomic_csv(pr, f_pr)
    save_atomic_csv(st, f_st)
    for p in glob.glob(os.path.join(OUT, "*.pdf")):
        os.remove(p)
    print(f"Chunk {ci}: docs {man.orig_file.nunique()} of {len(jobs)} | files {len(paths)} | PDFrate ok {len(vec)} | "
          f"Hidost ok {len(hok)} | generate failures {len(fails)}")

for ci in range(len(CHUNKS_B)):
    run_chunk(ci)

# ---------- merge ----------
pr_all = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f"{CK}/chunk_*_pdfrate.csv"))], ignore_index=True)
st_all = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(f"{CK}/chunk_*_status.csv"))], ignore_index=True)
st_files = st_all[st_all.out_file.fillna("") != ""]
PR = REF.merge(pr_all, on="out_file", how="inner")
save_atomic_csv(PR, f"{BIG}/pdfrate_features.csv")
save_atomic_csv(REF.merge(st_files, on="out_file", how="inner"), f"{BIG}/baseline_status.csv")
n_exp = len(REF[REF.orig_file.isin({j[0] for j in jobs_all})])
print(f"\n=== MERGED ===\nExpected files: {n_exp} | with PDFrate: {len(PR)} | with Hidost cache: {int(st_files.hidost_ok.sum())}"
      f" | with both: {int((st_files.pdfrate_ok & st_files.hidost_ok).sum())}")
if (~st_files.pdfrate_ok).any():
    print("PDFrate failure reasons:", st_files.loc[~st_files.pdfrate_ok, "pdfrate_err"].str[:60].value_counts().head(5).to_dict())

if TRIAL:
    # sanity on the trial documents: does each baseline see a difference between honest re-save (G1) and forgery (F1)?
    feat = [c for c in pr_all.columns if c != "out_file"]
    P1 = PR.set_index("out_file")
    d_pr, d_hd = [], []
    for o in PR.orig_file.unique():
        g1, f1 = o[:-4] + "__G1_resave.pdf", o[:-4] + "__F1_digit_resave.pdf"
        if g1 in P1.index and f1 in P1.index:
            d_pr.append(int((P1.loc[g1, feat].astype(float).fillna(-999.0) != P1.loc[f1, feat].astype(float).fillna(-999.0)).sum()))
        a, b = os.path.join(HC, os.path.realpath(os.path.join(OUT, g1)).lstrip("/")), os.path.join(HC, os.path.realpath(os.path.join(OUT, f1)).lstrip("/"))
        if os.path.exists(a) and os.path.exists(b):
            d_hd.append(open(a, "rb").read() != open(b, "rb").read())
    print(f"Trial: PDFrate features differing G1 vs F1 per document: {sorted(d_pr)}")
    print(f"Trial: Hidost cache differs G1 vs F1 in {sum(d_hd)} of {len(d_hd)} documents")
    sec_doc = (TIMES["generate"] + TIMES["pdfrate"] + TIMES["hidost"]) / max(1, TIMES["docs"])
    print(f"Trial timing per document: generate {TIMES['generate'] / max(1, TIMES['docs']):.1f}s | "
          f"PDFrate {TIMES['pdfrate'] / max(1, TIMES['docs']):.1f}s | Hidost {TIMES['hidost'] / max(1, TIMES['docs']):.1f}s")
    print(f"Projected FULL run: {len(KEEP)} documents x {sec_doc:.1f}s = {sec_doc * len(KEEP) / 3600:.1f} h "
          f"(+ ~0.5 h for CELL 41 rebuilding tools in a committed run). Kaggle limit: 12 h.")
else:
    with tarfile.open(f"{BIG}/hidost_cache.tar.gz", "w:gz") as tf:
        tf.add(HC, arcname="hidost_cache")
    shutil.rmtree(HC)                                      # keep only the archive (40k small files are slow to save)
    print(f"Hidost cache archived: {BIG}/hidost_cache.tar.gz ({os.path.getsize(f'{BIG}/hidost_cache.tar.gz') / 1e6:.1f} MB)")

json.dump({"trial": TRIAL, "docs": TIMES["docs"], "files": TIMES["files"], "pdfrate_files": len(PR),
           "hidost_files": int(st_files.hidost_ok.sum()), "seconds": {k: round(v, 1) for k, v in TIMES.items() if k not in ("files", "docs")}},
          open(f"{BIG}/cell44_report.json", "w"), indent=2)
print(f"\n=== CELL 44 {'TRIAL ' if TRIAL else ''}DONE === report: {BIG}/cell44_report.json")